In [1]:
import os
from pathlib import Path
import pymupdf4llm
from chonkie import RecursiveChunker, Pipeline

import re

from collections import Counter

In [2]:
PDF_DIR = Path("./raw_pdfs")
MARKDOWN_DIR = Path("./parsed_markdowns")
MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)

> # IMPORTANT Conversion exception
> 
> `Bowel_Cancer_UK_Colonic_Stenting_V2.1.pdf` has a faulty structure and cannot be parsed therefore was manually parsed, please download it from OneDrive.
> This feature might be added in the future using Image OCR. 

In [3]:
def convert_pdf_to_md(pdf_path: Path, output_dir: Path):
    try:
        if("raw_pdfs/Bowel_Cancer_UK_Colonic_Stenting_V2.1.pdf" in str(pdf_path)):
            raise Exception("Bowel_Cancer_UK_Colonic_Stenting_V2.1.pdf has a faulty structure and was manually parsed, please download it from OneDrive")
        markdown = pymupdf4llm.to_markdown(str(pdf_path))
        output_file = output_dir / (pdf_path.stem + ".md")
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(markdown)
        print(f"[OK] {pdf_path.name} -> {output_file.name}")
    except Exception as e:
        print(f"[FAIL] {pdf_path.name}: {e}")

In [4]:
pdf_files = list(PDF_DIR.glob("*.pdf"))

if not pdf_files:
    print("No PDF files found in raw_pdfs/")

for pdf in pdf_files:
    convert_pdf_to_md(pdf, MARKDOWN_DIR)

[FAIL] Bowel_Cancer_UK_Colonic_Stenting_V2.1.pdf: Bowel_Cancer_UK_Colonic_Stenting_V2.1.pdf has a faulty structure and was manually parsed, please download it from OneDrive
[OK] WOCN_Catheterization_Of_Urinary_Stoma_2018.pdf -> WOCN_Catheterization_Of_Urinary_Stoma_2018.md
[OK] NCPD_Nursing_Care_For_Patients_After_Urostomy_Surgery.pdf -> NCPD_Nursing_Care_For_Patients_After_Urostomy_Surgery.md
[OK] Bowel_Cancer_UK_Practical_Tips_Booklet_2025.pdf -> Bowel_Cancer_UK_Practical_Tips_Booklet_2025.md
[OK] Bowel_Cancer_UK_The_Facts_About_Bowel_Cancer_Easy_Read_Booklet.pdf -> Bowel_Cancer_UK_The_Facts_About_Bowel_Cancer_Easy_Read_Booklet.md
[OK] Bowel_Cancer_UK_About_Stoma_Reversal_2024.pdf -> Bowel_Cancer_UK_About_Stoma_Reversal_2024.md
[OK] NCCN_Colon_Cancer_Guideline_For_Patient_2025.pdf -> NCCN_Colon_Cancer_Guideline_For_Patient_2025.md
[OK] Bowel_Cancer_UK_Eating_Well.pdf -> Bowel_Cancer_UK_Eating_Well.md
[OK] ACS_Colorectal_Cancer_Fact_Sheet.pdf -> ACS_Colorectal_Cancer_Fact_Sheet.md


# Clean clean clean (preserve structure)


In [5]:
CLEANED_MARKDOWN_DIR = Path("./cleaned_markdowns")
CLEANED_MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
def load_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

In [33]:
HEADING_RE = re.compile(r'^\s*#{1,6}\s+')          # Markdown headings
LIST_RE = re.compile(r'^\s*([*\-]|\d+\.)\s+')     # bullets or numbered lists
BLANK_RE = re.compile(r'^\s*$')
DIVIDER_RE = re.compile(r'\s*---\s*')

TABLE_ROW_RE = re.compile(r'^\s*\|.*\|\s*$')
TABLE_SEPARATOR_RE = re.compile(r'^\s*\|?\s*:?-{2,}:?\s*(\|\s*:?-{2,}:?\s*)+\|?\s*$')

SENTENCE_END_RE = re.compile(r'[.!?:]\s*$')
LOWER_START_RE = re.compile(r'^[a-z(\[\{\)\]\}]')

In [34]:
def remove_headers_footers(text):
    lines = text.split("\n")
    
    cleaned = []
    for line in lines:
        # Page numberings
        if re.search(r'^(\*\*)?\d+\b(\*\*)?(?![\.\)-])', line):
            continue
        if len(line.strip()) < 3:
            cleaned.append(line)
            continue
        if line.isupper() and len(line.split()) > 5:
            # heuristic: large uppercase banner headers
            continue
        cleaned.append(line)
    
    return "\n".join(cleaned)

In [35]:
def is_protected(line):
    return (
        BLANK_RE.match(line) or
        LIST_RE.match(line) or
        TABLE_ROW_RE.match(line) or
        TABLE_SEPARATOR_RE.match(line)
    )

def detect_repeating_lines(text, min_length=5, top_k=20, freq_threshold=0.02):
    raw_lines = text.split("\n")
    
    # normalize but keep original for filtering
    lines = [l.strip() for l in raw_lines]
    
    # filter trivial + protected lines
    filtered = [
        l for l in lines
        if len(l) >= min_length and not is_protected(l)
    ]
    
    total_lines = len(filtered)
    counter = Counter(filtered)
    
    repeating = set()
    
    for line, count in counter.items():
        freq = count / total_lines if total_lines > 0 else 0
        
        if freq >= freq_threshold and count > 2:
            repeating.add(line)
    
    return repeating

In [36]:
def remove_repeating_lines(text):
    lines = text.split("\n")
    
    repeating_lines = detect_repeating_lines(text)
    
    cleaned = []
    for line in lines:
        stripped = line.strip()
        
        # remove statistically repeated lines
        if stripped in repeating_lines:
            continue
            
        cleaned.append(line)
    
    return "\n".join(cleaned)    

In [37]:
def normalize_whitespace(text):
    # remove starting spaces
    text = re.sub(r'^[ \t]+', '', text, flags=re.MULTILINE)
    
    # remove trailing spaces
    text = re.sub(r'[ \t]+$', '', text, flags=re.MULTILINE)
    
    # normalize multiple spaces
    text = re.sub(r'[ ]{2,}', ' ', text)
    
    # normalize excessive newlines (keep paragraph breaks)
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    return text

In [38]:
def remove_bold_asterisks(text):
    # Remove **bold** markers but keep inner text
    text = re.sub(r'\*\*(.*?)\*\*', r'\1', text)
    
    return text

In [39]:
def fix_broken_words(text):
    # join hyphenated line breaks: exam-\nple → example
    text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)
    text = re.sub(r'([a-zA-Z0-9]+)\s+(-\w+)', r'\1\2', text)
    
    return text

In [40]:
def normalize_bullets(text):
    # Replace common bullet characters with "- "
    text = re.sub(r'^\s*#*\s*([•▪◦●○]|(`o`))+\s*', '\n- ', text, flags=re.MULTILINE)
    
    return text

In [41]:
def is_table_line(line):
    return TABLE_ROW_RE.match(line) or TABLE_SEPARATOR_RE.match(line)

In [42]:
def merge_lines(text):
    lines = text.split("\n")
    merged = []
    in_table = False
    i = 0

    while i < len(lines):
        line = lines[i]
        stripped = line.strip()

        # --- TABLE HANDLING ---
        if is_table_line(line):
            merged.append(line)
            in_table = True
            i += 1
            continue
        else:
            if in_table:
                in_table = False

        # Divider lines (REMOVE)
        if DIVIDER_RE.match(line):
            i += 1
            continue

        # --- MODIFIED BLANK LINE HANDLING ---
        # Blank lines handling
        if BLANK_RE.match(line):
        
            # collapse extra blank lines around lists
            if merged:
                prev = merged[-1]
        
                # next non-empty line
                j = i + 1
                while j < len(lines) and BLANK_RE.match(lines[j]):
                    j += 1
        
                nxt = lines[j].strip() if j < len(lines) else ""
        
                # soft break inside sentence
                if (
                    j < len(lines)
                    and not SENTENCE_END_RE.search(prev)
                    and LOWER_START_RE.match(nxt)
                ):
                    i += 1
                    continue
        
                # list boundary normalization:
                # do NOT preserve blank line between list items
                if LIST_RE.match(prev) and LIST_RE.match(nxt):
                    i += 1
                    continue
        
            # preserve single paragraph break only
            if not merged or merged[-1] != "":
                merged.append("")
        
            i += 1
            continue
            
        # Headings → always new block
        if HEADING_RE.match(line):
            merged.append(stripped)
            i += 1
            continue

        # New list item → always new block
        if LIST_RE.match(line):
            merged.append(stripped)
            i += 1
            continue

        # Continuation logic
        if merged:
            prev = merged[-1]

            # do not merge into table
            if is_table_line(prev):
                merged.append(stripped)
                i += 1
                continue

            # previous is list item → append continuation
            if LIST_RE.match(prev):
                merged[-1] = prev + " " + stripped
                i += 1
                continue

            # previous is normal paragraph → merge ONLY if not sentence end
            if not HEADING_RE.match(prev) and not SENTENCE_END_RE.search(prev) and LOWER_START_RE.match(stripped):
                merged[-1] = prev + " " + stripped
                i += 1
                continue

        # fallback
        merged.append(stripped)
        i += 1

    return "\n".join(merged)

In [43]:
HEADING_RE = re.compile(r'^\s*(#{1,6})\s+(.*)')
MAX_HEADER_WORDS = 10  # prevent paragraphs becoming headers

def clean_and_normalize_markdown(text):
    lines = text.split("\n")
    processed_lines = []
    
    # State tracking
    seen_titles = set()
    prev_level = 0
    i = 0

    while i < len(lines):
        line = lines[i].strip()
        match = HEADING_RE.match(line)

        if not match:
            processed_lines.append(lines[i]) # Keep original indentation for non-headers
            i += 1
            continue

        # --- 1. MERGING LOGIC (from normalize_headers) ---
        hashes, content = match.groups()
        combined_content = content
        j = i + 1
        
        while j < len(lines):
            next_line = lines[j].strip()
            next_match = HEADING_RE.match(next_line)
            
            if next_match:
                _, next_content = next_match.groups()
                combined_content += " " + next_content
                j += 1
            elif next_line and len(next_line.split()) <= 6:
                combined_content += " " + next_line
                j += 1
            else:
                break

        # --- 2. VALIDATION (MAX_HEADER_WORDS & fake headers) ---
        words = combined_content.split()
        
        # Check if it has content following it (from remove_fake_headers)
        has_following_content = False
        k = j
        while k < len(lines):
            if HEADING_RE.match(lines[k].strip()):
                break
            if lines[k].strip():
                has_following_content = True
                break
            k += 1

        # If it's too long or has no content following it, treat as normal text
        if len(words) > MAX_HEADER_WORDS:
            processed_lines.append(combined_content)
            i = j
            continue
            
        if not has_following_content:
            i=j
            continue

        # --- 3. LEVEL NORMALIZATION (from normalize_header_levels) ---
        current_level = len(hashes)
        title_key = combined_content.lower().strip()

        # Prevent jumps (Rule 1)
        if prev_level and current_level > prev_level + 1:
            current_level = prev_level + 1

        # Demote repeats (Rule 2)
        if title_key in seen_titles:
            current_level = min(current_level + 1, 6)
        else:
            seen_titles.add(title_key)

        # Avoid multiple H1s (Rule 3)
        if current_level == 1 and prev_level != 0:
            current_level = 2

        # Finalize header
        processed_lines.append("#" * current_level + " " + combined_content.strip())
        prev_level = current_level
        i = j

    return "\n".join(processed_lines)

In [44]:
def clean_text(text):
    text = normalize_whitespace(text)
    text = remove_bold_asterisks(text)
    text = fix_broken_words(text)
    text = normalize_bullets(text)
    text = merge_lines(text)
    text = clean_and_normalize_markdown(text)
    text = remove_repeating_lines(text)
    text = remove_headers_footers(text)
    text = normalize_whitespace(text)
    return text

In [45]:
for file_path in MARKDOWN_DIR.glob("*.md"):
    raw = load_file(file_path)
    cleaned = clean_text(raw)
    
    output_path = CLEANED_MARKDOWN_DIR / (file_path.stem + ".md")
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(cleaned)

# Chunking
## I used [Chonkie](https://github.com/chonkie-inc/chonkie)

## TODO: Try NeuralChunker on better machine :(
> My Mac Intel cannot handle it

In [23]:
from chonkie import NeuralChunker

# Basic initialization with default parameters
chunker = NeuralChunker(
    model="mirth/chonky_modernbert_base_1",  # Default model
    device_map="cpu",                        # Device to run the model on ('cpu', 'cuda', etc.)
    min_characters_per_chunk=10,             # Minimum characters for a chunk
)

/Users/lethienson/Desktop-Files/codean/capstone-mobicure-cappuccino/DocumentsChunking/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


ImportError: 
AutoModelForTokenClassification requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.


## Semantic Pipeline (**Chosen**)
- Used with Embeddings and Overlap refinement

In [63]:
from chonkie import OverlapRefinery
from chonkie.types import RecursiveRules, RecursiveLevel

rules = RecursiveRules(
    levels=[
        # Paragraph-level first
        RecursiveLevel(
            delimiters=["\n\n"],
            include_delim="prev"
        ),

        # Sentence-level fallback
        RecursiveLevel(
            delimiters=[". ", "! ", "? "],
            include_delim="prev"
        )
    ]
)

In [64]:
# Process all markdown files in a directory
doc = (Pipeline()
    .fetch_from("file", path="./cleaned_markdowns/Bowel_Cancer_UK_Colonic_Stenting_V2.1.md")
    .process_with("markdown")
    .chunk_with("semantic",
                threshold=0.7,
                chunk_size=500,
                similarity_window=6,
                skip_window=0,
                min_sentences_per_chunk=2,
                delim=[". ", "!", "?", "\n\n", "#", "##", "###", "####"],
                )
    .refine_with(
        "overlap",
        tokenizer="word",
        context_size=0.12,
        mode="recursive",
        method="prefix",
        rules=rules,
        merge=True,
        inplace=True)
    .refine_with("embeddings", embedding_model="minishlab/potion-base-32M")
    .run())

# Process each document
print(f"Document has {len(doc.chunks)} chunks")
for chunk in doc.chunks:
    print(f"\t Chunk has {chunk.token_count} tokens ")
    print(chunk.text)
    print("\n\n------------------------------------------------------------\n\n")

Fetching 10 files: 100%|████████████████████████████| 10/10 [00:00<00:00, 17667.67it/s]


Document has 17 chunks
	 Chunk has 141 tokens 
# About Colonic Stenting

This factsheet is about having a colonic stent fitted. It explains why you might have a stent fitted, what’s involved, and the possible complications.

## What is a colonic stent?

A colonic stent is a small flexible tube that is inserted into your bowel to keep the bowel open when it has become blocked (obstructed) or partly blocked by bowel cancer.

The stent is made of a metal mesh, sometimes coated with silicone. A stent doesn’t treat the cancer but it can help to manage the symptoms of the blockage.

## Why are stents used?

If your bowel has become blocked, it may be difficult for your bowel movements to pass through. 


------------------------------------------------------------


	 Chunk has 252 tokens 
If your bowel has become blocked, it may be difficult for your bowel movements to pass through. This can cause pain, cramping, bloating and/or sickness.

A stent can relieve these symptoms by widening the 

# Markdown REcipe (NAhhhhh not good, chunks too big, the parsing were inconsistent)

In [48]:
# Load markdown processing recipe
pipeline = Pipeline.from_recipe("markdown")

# Run with your content
doc = pipeline.fetch_from("file", path="cleaned_markdowns/Bowel_Cancer_UK_Colonic_Stenting_V2.1.md").run()

print(f"Document has {len(doc.chunks)} chunks")
for chunk in doc.chunks:
    print(f"\t Chunk has {chunk.token_count} tokens ")
    print(chunk.text)
    print("\n\n------------------------------------------------------------\n\n")

2026-05-07 08:15:07,971 | WARNING  | chonkie.chunker.code:__init__:70 - The language is set to `auto`. This would adversely affect the performance of the chunker. Consider setting the `language` parameter to a specific language to improve performance.


Document has 5 chunks
	 Chunk has 2122 tokens 
# About Colonic Stenting

This factsheet is about having a colonic stent fitted. It explains why you might have a stent fitted, what’s involved, and the possible complications.

## What is a colonic stent?

A colonic stent is a small flexible tube that is inserted into your bowel to keep the bowel open when it has become blocked (obstructed) or partly blocked by bowel cancer.

The stent is made of a metal mesh, sometimes coated with silicone. A stent doesn’t treat the cancer but it can help to manage the symptoms of the blockage.

## Why are stents used?

If your bowel has become blocked, it may be difficult for your bowel movements to pass through. This can cause pain, cramping, bloating and/or sickness.

A stent can relieve these symptoms by widening the bowel to open and unblock it. This allows your bowel to empty more easily again.

Stents can be temporary or permanent.

- You might be offered a temporary stent if you are going to have